# SCA AutoFix Agent - MCP Tool-Calling OpenAI Agent

This notebook creates and deploys an AI agent that connects to the **SCA AutoFix MCP Server** to analyze vulnerable dependencies and generate fix recommendations.

## What This Agent Does

The agent connects to our custom MCP server (`sca-autofix-agent`) which provides tools for:

1. **Vulnerability Scanning** - Get vulnerable repos from Unity Catalog
2. **API Diff Analysis** - Compare library versions for breaking changes  
3. **Code Analysis** - Find library usages in repositories
4. **Impact Assessment** - Match breaking changes with actual code usage
5. **Fix Generation** - Generate code patches using LLM

## Architecture

```
UI (Streamlit) → Agent Endpoint → This Agent → SCA MCP Server → Tools
                                      ↓
                              Databricks LLM
```

## Prerequisites
- SCA AutoFix MCP Server deployed as a Databricks App
- Service Principal with OAuth M2M credentials
- Unity Catalog tables configured

In [0]:
%pip install --force-reinstall  databricks-openai databricks-agents uv
dbutils.library.restartPython()

## Define the SCA AutoFix Agent

The agent code is written to a file using `%%writefile` for MLflow logging and deployment.

**Key Components:**

1. **MCP Server Connection** - Connects to our custom SCA AutoFix MCP server (Databricks App)
2. **System Prompt** - Specialized for vulnerability analysis and code fixes
3. **Tool Execution** - Handles the 14 SCA tools via MCP protocol
4. **Streaming Response** - Compatible with Mosaic AI ResponsesAgent interface

**Available MCP Tools:**
- `get_vulnerable_repos` - List repos with vulnerabilities
- `get_repo_details` - Get details for a specific repo
- `analyze_library_usage` - Find library usages in code
- `get_api_diff` - Compare library versions
- `match_breaking_changes` - Match API changes with code
- `generate_code_fix` - Generate fix using LLM
- `write_analysis_result` - Save results to Unity Catalog
- And more...

In [0]:
%%writefile agent.py

import mlflow
from mlflow.entities import SpanType
from typing import Any, Generator
from uuid import uuid4
from databricks_openai import McpServerToolkit, ToolInfo
from databricks.sdk import WorkspaceClient
from mlflow.pyfunc import ResponsesAgent
from databricks_openai import DatabricksOpenAI
import openai
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
import nest_asyncio
import json
import os

nest_asyncio.apply()

############################################
## SCA AutoFix Agent Configuration
############################################

# LLM endpoint for code generation
LLM_ENDPOINT_NAME = os.environ.get("LLM_ENDPOINT", "databricks-meta-llama-3-1-70b-instruct")

# SCA AutoFix MCP Server URL (deployed as Databricks App)
SCA_MCP_SERVER_URL = os.environ.get("SCA_MCP_SERVER_URL", "")

# System prompt for SCA vulnerability analysis
SYSTEM_PROMPT = """You are an expert SCA (Software Composition Analysis) assistant that helps developers fix vulnerable dependencies.

Your capabilities:
1. Identify vulnerable repositories and libraries from Unity Catalog
2. Analyze API differences between library versions
3. Find where libraries are used in code
4. Match breaking API changes with actual code usage
5. Generate safe upgrade patches with minimal code changes

When analyzing a vulnerability upgrade:
1. First understand the current state (repo, library, versions)
2. Get the API diff to see what changed between versions
3. Analyze the codebase to find usages of changed APIs
4. Match the changes to identify impacted code
5. Generate targeted fixes only for affected code

Always explain your reasoning and provide safe, tested upgrade paths.
Prefer non-breaking changes when possible. Flag any manual review requirements.
"""

############################################
## MCP Server Connection Setup
############################################

workspace_client = WorkspaceClient()
host = workspace_client.config.host

# ---------------------------------------------------------------------------
# Custom MCP Server Setup for SCA AutoFix
# ---------------------------------------------------------------------------
# The SCA MCP server is hosted as a Databricks App
# Requires OAuth M2M authentication with a Service Principal

# Get credentials from environment (set via secrets in deployment)
DATABRICKS_CLIENT_ID = os.environ.get("DATABRICKS_CLIENT_ID", "")
DATABRICKS_CLIENT_SECRET = os.environ.get("DATABRICKS_CLIENT_SECRET", "")

# Create workspace client with OAuth for custom MCP server
if DATABRICKS_CLIENT_ID and DATABRICKS_CLIENT_SECRET and SCA_MCP_SERVER_URL:
    custom_mcp_workspace_client = WorkspaceClient(
        host=host,
        client_id=DATABRICKS_CLIENT_ID,
        client_secret=DATABRICKS_CLIENT_SECRET,
        auth_type="oauth-m2m",
    )
    mcp_servers = [
        McpServerToolkit(url=SCA_MCP_SERVER_URL, workspace_client=custom_mcp_workspace_client),
    ]
else:
    # Fallback: use managed MCP server with python_exec for testing
    print("⚠️ SCA MCP Server not configured - using system.ai.python_exec for testing")
    mcp_servers = [
        McpServerToolkit(url=f"{host}/api/2.0/mcp/functions/system/ai"),
    ]


############################################
## SCA AutoFix Agent Class
############################################

class MCPToolCallingAgent(ResponsesAgent):
    """
    SCA AutoFix Agent that connects to MCP tools for vulnerability analysis.
    """
    
    def __init__(self, llm_endpoint: str, mcp_servers: list[McpServerToolkit]):
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client = DatabricksOpenAI()
        self.mcp_servers = mcp_servers
        self.tools_dict = {}

        for mcp_server in mcp_servers:
            tool_infos = mcp_server.get_tools()
            for tool_info in tool_infos:
                if tool_info.name in self.tools_dict:
                    raise ValueError(
                        f"Tool Name {tool_info.name} already exists. "
                        f"For MCP Server: {mcp_server.name or mcp_server.url}, "
                        "specify a new mcp server name to make the tool names unique."
                    )
                self.tools_dict[tool_info.name] = tool_info
        
        print(f"✅ Agent initialized with {len(self.tools_dict)} tools")
        for name in self.tools_dict.keys():
            print(f"   - {name}")

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        return self.tools_dict[tool_name].execute(**args)

    @mlflow.trace(span_type=SpanType.LLM)
    def call_llm(
        self, messages: list[dict[str, Any]]
    ) -> Generator[dict[str, Any], None, None]:
        # Prepend system prompt
        full_messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
        
        for chunk in self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=to_chat_completions_input(full_messages),
            tools=[tool.spec for tool in self.tools_dict.values()],
            stream=True,
        ):
            yield chunk.to_dict()

    def handle_tool_call(
        self, tool_call: dict[str, Any], messages: list[dict[str, Any]]
    ) -> ResponsesAgentStreamEvent:
        """
        Execute tool calls, add them to the running message history, 
        and return a ResponsesStreamEvent with tool output.
        """
        if tool_call["arguments"]:
            args = json.loads(tool_call["arguments"])
        else:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_call_output = self.create_function_call_output_item(
            tool_call["call_id"], result
        )
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(
            type="response.output_item.done", item=tool_call_output
        )

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 10,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role", None) == "assistant":
                return
            elif last_msg.get("type", None) == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item(
                "Max iterations reached. Stopping.", str(uuid4())
            ),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(
            output=outputs, custom_outputs=request.custom_inputs
        )

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        messages = [i.model_dump() for i in request.input]
        yield from self.call_and_run_tools(messages)


# Enable MLflow autologging for tracing
mlflow.openai.autolog()

# Create the agent instance
AGENT = MCPToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, mcp_servers=mcp_servers)
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since we manually traced methods within `ResponsesAgent`, you can view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
# ==============================================================================
# TODO: ONLY UNCOMMENT AND EDIT THIS SECTION IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
#       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
# ==============================================================================

# # Set your Databricks client ID and client secret for service principal authentication.
# DATABRICKS_CLIENT_ID = "<YOUR_CLIENT_ID>"
# client_secret_scope_name = "<YOUR_SECRET_SCOPE>"
# client_secret_key_name = "<YOUR_SECRET_KEY_NAME>"

# # Load your service principal credentials into environment variables
# os.environ["DATABRICKS_CLIENT_ID"] = DATABRICKS_CLIENT_ID
# os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=client_secret_scope_name, key=client_secret_key_name)

In [0]:
from agent import AGENT

# Test basic agent functionality
result = AGENT.predict({
    "input": [
        {"role": "user", "content": "List the vulnerable repositories with Critical priority"}
    ]
})
print(result.model_dump(exclude_none=True))

In [0]:
# Test streaming for a full analysis workflow
for chunk in AGENT.predict_stream({
    "input": [
        {"role": "user", "content": """
Analyze the jackson-databind vulnerability in java-goof-todolist repo.
1. Get the API diff from version 2.9.8 to 2.15.0
2. Find jackson-databind usages in the repository
3. Match the breaking changes with the code
4. Generate a fix recommendation
"""}
    ]
}):
    print(chunk.model_dump(exclude_none=True))

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).



In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME, SCA_MCP_SERVER_URL
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from pkg_resources import get_distribution

# Resources required by the agent
resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
]

# Add SCA MCP server if configured (the Databricks App provides the tools)
# Note: Custom MCP servers don't need DatabricksFunction resources
# since they're accessed via HTTP, not UC functions

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="sca-autofix-agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            "backoff",
            f"mlflow=={get_distribution('mlflow').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-openai=={get_distribution('databricks-openai').version}"
        ]
    )
    print(f"✅ Agent logged: {logged_agent_info.model_uri}")

## Evaluate the agent with Agent Evaluation

Use Mosaic AI Agent Evaluation to evalaute the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.
See Databricks documentation ([AWS]((https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety

# Evaluation dataset for SCA AutoFix scenarios
eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "List all Critical priority vulnerable repositories"}]},
        "expected_response": "Found vulnerable repositories with Critical priority including jackson-databind, log4j-core vulnerabilities.",
    },
    {
        "inputs": {"input": [{"role": "user", "content": "What are the breaking changes between jackson-databind 2.9.8 and 2.15.0?"}]},
        "expected_response": "The API diff shows breaking changes in ObjectMapper configuration and serialization features.",
    },
    {
        "inputs": {"input": [{"role": "user", "content": "Generate a fix for jackson-databind upgrade in java-goof repo"}]},
        "expected_response": "Generated code patch to update ObjectMapper usage to be compatible with jackson-databind 2.15.0.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],
)

# Review the evaluation results in the MLfLow UI

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}]},
    env_manager="uv",
)

## Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Unity Catalog model registration
# TODO: Update these values for your workspace
catalog = "ing_hackathon"
schema = "ing_hackathon"
model_name = "sca_autofix_agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, 
    name=UC_MODEL_NAME
)
print(f"✅ Registered model: {UC_MODEL_NAME} v{uc_registered_model_info.version}")

## Deploy the agent

In [0]:
from databricks import agents

# Deploy the agent to a serving endpoint
# The endpoint name will be: fix_advisor_agent_endpoint (matching UI config)

# Service Principal credentials for OAuth M2M to access the SCA MCP Server
# These should be stored in Databricks Secrets
client_secret_scope_name = "sca-autofix"  # TODO: Create this secret scope
client_secret_key_name = "sp-client-secret"

# SCA MCP Server URL (your Databricks App URL)
SCA_MCP_APP_URL = "https://<workspace-url>/apps/sca-autofix-agent"  # TODO: Update with actual app URL

agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version,
    endpoint_name="fix_advisor_agent_endpoint",  # Matches UI config
    environment_vars={
        # SCA MCP Server connection
        "SCA_MCP_SERVER_URL": SCA_MCP_APP_URL,
        
        # OAuth M2M credentials for the Service Principal
        "DATABRICKS_CLIENT_ID": "<YOUR_SERVICE_PRINCIPAL_CLIENT_ID>",  # TODO: Fill in
        "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}",
        
        # LLM endpoint override (optional)
        "LLM_ENDPOINT": "databricks-meta-llama-3-1-70b-instruct",
    },
    tags={"app": "sca-autofix", "version": "1.0"},
    deploy_feedback_model=False
)

print(f"✅ Agent deployed to endpoint: fix_advisor_agent_endpoint")

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See docs ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)) for details